Ноутбук отвечает за дообучение multilingual-e-base. Эта модель использовалась как более легкий dense retrieval baseline для сравнения с PromptRetriever.


In [ ]:
!nvidia-smi
!pip -q install transformers accelerate sentencepiece protobuf datasets tqdm

import os, json, random, time, math, gc
from collections import Counter
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

TRAIN_FILE = globals().get(
    "TRAIN_FILE",
    "/kaggle/input/datasets/sukiss/prmptr/tevatron_ru_promptriever_train (2).jsonl"
)

MODEL_NAME = "intfloat/multilingual-e5-base"
OUTPUT_DIR = "/kaggle/working/e5_multilingual_base_ru_retriever"
ARCHIVE_PATH = "/kaggle/working/e5_multilingual_base_ru_retriever.tar.gz"

SEED = 42
EPOCHS = 1
BATCH_SIZE = 8
GRAD_ACCUM = 2
MAX_NEGATIVES = 5

QUERY_MAX_LEN = 192
PASSAGE_MAX_LEN = 192

LR = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
SAVE_EVERY = 100
MAX_STEPS = None

USE_ONLY_INSTRUCTION_ROWS = False

random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "Need GPU"

os.makedirs(OUTPUT_DIR, exist_ok=True)

class RetrievalTrainDataset(Dataset):
    def __init__(self, path):
        self.items = []
        stats = Counter()

        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                stats["seen"] += 1
                item = json.loads(line)

                meta = item.get("metadata", {})
                if USE_ONLY_INSTRUCTION_ROWS and meta.get("has_instruction") is not True:
                    stats["skip_not_instruction"] += 1
                    continue

                query = (item.get("query") or "").strip()
                positives = item.get("positive_passages") or []
                negatives = item.get("negative_passages") or []

                if not query or len(positives) < 1 or len(negatives) < 1:
                    stats["skip_bad_fields"] += 1
                    continue

                pos = (positives[0].get("text") or "").strip()
                negs = [(x.get("text") or "").strip() for x in negatives]
                negs = [x for x in negs if x]

                if not pos or not negs:
                    stats["skip_empty_text"] += 1
                    continue

                self.items.append({
                    "query": query,
                    "positive": pos,
                    "negatives": negs,
                    "query_id": item.get("query_id"),
                    "has_instruction": meta.get("has_instruction") is True,
                })
                stats["kept"] += 1

        print("Dataset stats:", stats)
        print("Rows:", len(self.items))
        print("Instruction rows:", sum(1 for x in self.items if x["has_instruction"]))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        negs = item["negatives"]
        if len(negs) > MAX_NEGATIVES:
            negs = random.sample(negs, MAX_NEGATIVES)

        return {
            "query": item["query"],
            "passages": [item["positive"]] + negs,
        }

def collate_fn(batch):
    queries = []
    passages = []
    labels = []

    offset = 0
    for item in batch:
        queries.append(item["query"])
        labels.append(offset)
        passages.extend(item["passages"])
        offset += len(item["passages"])

    return queries, passages, torch.tensor(labels, dtype=torch.long)

dataset = RetrievalTrainDataset(TRAIN_FILE)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.train()

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

def average_pool(last_hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_states.size()).float()
    masked = last_hidden_states * mask
    return masked.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

def encode_texts(texts, max_len, is_query):
    prefix = "query: " if is_query else "passage: "
    texts = [prefix + x for x in texts]

    batch = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}

    out = model(**batch)
    emb = average_pool(out.last_hidden_state, batch["attention_mask"])
    return F.normalize(emb, p=2, dim=-1)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

steps_per_epoch = math.ceil(len(loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
if MAX_STEPS is not None:
    total_steps = min(total_steps, MAX_STEPS)

warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

scaler = torch.cuda.amp.GradScaler()

global_step = 0
running_loss = 0.0
start = time.time()

optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):
    pbar = tqdm(loader, desc=f"epoch {epoch + 1}")

    for step, (queries, passages, labels) in enumerate(pbar):
        labels = labels.to(DEVICE)

        with torch.cuda.amp.autocast(dtype=torch.float16):
            q_emb = encode_texts(queries, QUERY_MAX_LEN, is_query=True)
            p_emb = encode_texts(passages, PASSAGE_MAX_LEN, is_query=False)

            scores = torch.matmul(q_emb, p_emb.T) / 0.01
            loss = F.cross_entropy(scores, labels)
            loss = loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            global_step += 1
            avg_loss = running_loss / GRAD_ACCUM
            running_loss = 0.0

            pbar.set_postfix({
                "loss": round(avg_loss, 4),
                "global_step": global_step,
                "lr": scheduler.get_last_lr()[0],
            })

            if global_step % SAVE_EVERY == 0:
                ckpt = os.path.join(OUTPUT_DIR, f"checkpoint-{global_step}")
                os.makedirs(ckpt, exist_ok=True)
                model.save_pretrained(ckpt)
                tokenizer.save_pretrained(ckpt)
                print("Saved checkpoint:", ckpt)

            if MAX_STEPS is not None and global_step >= MAX_STEPS:
                break

    if MAX_STEPS is not None and global_step >= MAX_STEPS:
        break

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

elapsed = time.time() - start

print("Saved final model:", OUTPUT_DIR)
print("Global steps:", global_step)
print("Elapsed hours:", elapsed / 3600)

!tar -czf "$ARCHIVE_PATH" -C "/kaggle/working" "e5_multilingual_base_ru_retriever"
print("Archive:", ARCHIVE_PATH)
